# Running a Data Genomics Pipeline Using Kubernetes
This notebook will walk through running a data genomics pipline using Kubernetes.

The *Escherichia coli* dataset used is part of a long-term evolution experiment led by [Richard Lenski]. 

The cells below are completing the steps from the [Data Carpentry - Data Wrangling and Processing for Genomics](https://datacarpentry.github.io/wrangling-genomics/)

# Step 0: Prerequisites

1. You must have an NRP account
2. You must have been added to a Nautilus namespace
3. You must have your NRP config in the `~/.kube` directory. There is a notebook to assist you [here](./NautilusConfigSetup.ipynb).
4. You must have a PVC on the Nautilus cluster in your assigned namespace



# Step 1: Downloading the data

Our first step is to download the data to a PVC for analysis.

In [2]:
from jinja2 import Template

# read in the template
with open('../yaml/data-download-job.yaml') as file_:
    template = Template(file_.read())

Replace the arguments to the `render` function with the appropriate values:
#### Items to change:
PVCNAME needs to be changed to a string value

JOBNAME needs to be changed to a string value

In [3]:
# render the job spec
job_spec = template.render(
    job_name="download-data",
    pvc_name="cbohn-pvc",
)

# print the job spec
print(job_spec)

apiVersion: batch/v1
kind: Job
metadata:
  name: download-data
spec:
  template:
    spec:
      containers:
      - name: download-data
        image: alpine/curl
        command: ["/bin/sh", "-c"]
        args:
        - |
          
          # 1. Navigate to your PVC mount directory
          cd /data && \
          
          # 2. Download the files directly into the PVC
          curl -O ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR258/004/SRR2589044/SRR2589044_1.fastq.gz && \
          curl -O ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR258/004/SRR2589044/SRR2589044_2.fastq.gz && \
          curl -O ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR258/003/SRR2584863/SRR2584863_1.fastq.gz && \
          curl -O ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR258/003/SRR2584863/SRR2584863_2.fastq.gz && \
          curl -O ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR258/006/SRR2584866/SRR2584866_1.fastq.gz && \
          curl -O ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR258/006/SRR2584866/SRR2584866_2.fastq.gz && \

        

Now, let's save it to disk:

In [4]:
with open("./data_download_job.yml", "w") as file:
    file.write(job_spec)

Run the cell below to download our data:

In [5]:
! kubectl create -f ./data_download_job.yml

job.batch/download-data created


Let's check the output of the job's pod to see if our data has been downloaded. Change `PODNAME` below to the correct pod name:

#### Items to change:
PODNAME needs to be changed

In [6]:
! kubectl logs --tail=5 download-data-nlmkx

100 308.8M 100 308.8M   0      0  5.66M      0   00:54   00:54         20.83M
  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed
100 295.9M 100 295.9M   0      0  1.75M      0   02:48   02:48          1.71M
Data Downloaded


In [8]:
! kubectl exec pod-name-sso -- ls /data 

SRR2584863_1.fastq
SRR2584863_2.fastq
SRR2584866_1.fastq
SRR2584866_2.fastq
SRR2589044_1.fastq
SRR2589044_2.fastq


## Step 2: Run the FASTQC tool

This will generate our quality control reports for our dataset. 

In [10]:
# read in the template
with open('../yaml/run-fastq.yaml') as file_:
    template = Template(file_.read())

Replace the arguments to the `render` function with the appropriate values:
#### Items to change:
PVCNAME needs to be changed to a string value

JOBNAME needs to be changed to a string value

In [11]:
# render the job spec
job_spec = template.render(
    job_name="run-fastq",
    pvc_name="cbohn-pvc",
)

# print the job spec
print(job_spec)

apiVersion: batch/v1
kind: Job
metadata:
  name: run-fastq
spec:
  template:
    spec:
      # Workaround for re-using the image
      securityContext:
        runAsUser: 0
        
      containers:
      - name: fastq
        image: gitlab-registry.nrp-nautilus.io/gp-engine/dc-genomics
        command: ["/bin/sh", "-c"]
        args:
        - |
          cd /data && \ 
        
          for input_file in *.fastq; do
            echo $input_file
            head -n4 $input_file
            fastqc $input_file
          done
        resources:
           limits:
             memory: 12Gi
             cpu: 4
           requests:
             memory: 12Gi
             cpu: 4
        volumeMounts:
        - name: cbohn-pvc
          mountPath: /data
      volumes:
      - name: cbohn-pvc
        persistentVolumeClaim:
          claimName: cbohn-pvc
      restartPolicy: Never
  backoffLimit: 1


Now let's save it to disk

In [12]:
with open("./run_fastq.yml", "w") as file:
    file.write(job_spec)

Run the cell below to process our data in a job:

In [13]:
! kubectl create -f ./run_fastq.yml

job.batch/run-fastq created


Let's check the output of the job's pod to see if our data has been downloaded. Change `PODNAME` below to the correct pod name:

#### Items to change:
PODNAME needs to be changed

In [ ]:
! kubectl logs --tail=5 PODNAME

In [ ]:
! kubectl exec PODNAME -- ls /data/*.html

## Step 3: Copy our data to view the results

We will now download our data from the pvc using our pod for copying data from earlier.

In [ ]:
! kubectl exec PODNAME -- /bin/bash -c "cd /data && tar czf fastq_html.tar.gz *.html"

In [ ]:
! kubectl cp PODNAME:/data/fastq_html.tar.gz ./

In [ ]:
! tar zxf fastq_html.tar.gz